**imports**
- sys.path is the list of folders where Python looks for modules
- The notebook runs from within the `notebooks/` directory, and `src/` is one level up (`..`), so that folder needs to be added.

In [1]:
import sys
sys.path.append("..")          # the notebook runs inside notebooks/; this lets Python find src/

import numpy as np
import pandas as pd
from src.cashflow import load_constants
from src.cashflow import run_trajectory
from src.bootstrap import block_bootstrap
from src.bootstrap import summarize_trajectory
from src.bootstrap import simulate

%load_ext autoreload
%autoreload 2

pd.set_option("display.float_format", "{:,.4f}".format)


**Comparison: real vs theoretical autocorrelation**
- Real autocorrelation: e_base.autocorr(lag=k)
- theoretical autocorrelation: 0.64 ** k

In [2]:
#1. Upload csv file
df = pd.read_csv(r"../data/processed/annual_energy.csv")
e_base = df ['energy_base_gwh']

#2. Create the list with rows in 'for' loop
rows = []
for k in range(1, 9):
    rows.append({
        "lag": k, 
        "real": e_base.autocorr(lag=k),
        "ar1": 0.64 ** k})

#3. Create and show dataframe
acf = pd.DataFrame(rows)
acf

,lag,real,ar1
0,1,0.6445,0.6400
1,2,0.4662,0.4096
2,3,0.3525,0.2621
3,4,0.3216,0.1678
4,5,0.2534,0.1074
5,6,0.2408,0.0687
6,7,0.2138,0.0440
7,8,0.2083,0.0281


**Interpretation**:
- The “each year depends only on the previous one” model underestimates the river’s memory. When accounting for actual autocorrelation, the drop below 0.236 does not occur until 6 to 7 years later, not at 3 or 4 years. The river possesses a slow, multi-year component that the 0.64^k term fails to capture.

**Choosing the new L value**:
We need to consider that:
- The 0.236 band is thinner than it looks.
- There is a third, automatic criterion, which is the standard one. The Politis and White method estimates the optimal block length directly from the series.

In [3]:
from arch.bootstrap import optimal_block_length
optimal_block_length(df['energy_base_gwh'])


,stationary,circular
energy_base_gwh,6.9792,7.9891


**Result summary to choose L**
| Method | Suggested L | Observation |
| :--- | :---: | :---: |
| n^(1/3) | 4   | This method does not have any constant that depends of the data serie |
| Actual autocorrelation vs. band | 6 - 7 | The band is not that precise |
| Politis-White (circular) | 8 | Published method, automatic, without estimated ranges |

**Simulate function for Base case**

In [4]:
semi = pd.read_csv("../data/processed/semiannual_energy.csv")
table = semi.pivot(index="hydro_year", columns="half", values="energy_base_gwh").to_numpy()

# Parameters
n_records = table.shape[0]                           # (69, 2): one row per hydrological year, columns S1 and S2
n_years = 30 
L=8 

# Random number generator
rng = np.random.default_rng(seed=42)

# Run a complete trajectory
    # Positions
positions = block_bootstrap(n_records, n_years, L, rng)
    # Extract the sorted rows
sorted_years_table = table[positions]
    # flatten row pwe row to obtain the cronological serie of 60 semesters
energy_trajectory = sorted_years_table.ravel()

# Execute funtion run_trajectory
c = load_constants("../model/hydro_pf_model_v1.xlsx")
result = run_trajectory(energy_trajectory, c )

print("BASE CASE WITH L=8:")
print(f"Table extracted per year: {sorted_years_table.shape}")  # Should be (30, 2)
print(f"Length of the final energy vector: {len(energy_trajectory)}")  # Should be 60
print("\nFirst 6 values of the vector (S1 year  1, S2 year 1, S1 year 2...):")
print(energy_trajectory[:6])

print(f"Minimum DSCR (12m) = ",(result["dscr_12m"][c["repayment_flag"] == 1].min()))
print(f"Equity IRR = " , result["equity_irr"])
print(f"Quantity of lockups = ", result["lockup"].sum())
print(f"Quantity of defaults = ", result["default"].sum())

BASE CASE WITH L=8:
Table extracted per year: (30, 2)
Length of the final energy vector: 60

First 6 values of the vector (S1 year  1, S2 year 1, S1 year 2...):
[34.72624783 14.31234707 33.77413039 18.37214137 29.34512023 14.35140804]
Minimum DSCR (12m) =  0.775677637420131
Equity IRR =  0.03045651571905328
Quantity of lockups =  2.0
Quantity of defaults =  2.0


In [5]:

sims = simulate(table, c, 1000, 8, np.random.default_rng(42))
sims.mean()

min_dscr_12m            1.1653
any_lockup              0.5950
lockup_semesters        1.3080
any_default             0.2600
dsra_depleted           0.3990
sponsor_support    35,982.2829
equity_irr              0.0306
dtype: float64

**Simulate function for Tariff = 100.77 case**

In [6]:
# --------------- tariff100_p90 CASE ------------------
# Execute funtion run_trajectory
c = load_constants("../tests/fixtures/tariff100_p90.xlsx")
result = run_trajectory(energy_trajectory, c )

print(f"Table extracted per year: {sorted_years_table.shape}")  # Should be (30, 2)
print(f"Length of the final energy vector: {len(energy_trajectory)}")  # Should be 60
print("\nFirst 6 values of the vector (S1 year  1, S2 year 1, S1 year 2...):")
print(energy_trajectory[:6])

print(f"Minimum DSCR (12m) = ",(result["dscr_12m"][c["repayment_flag"] == 1].min()))
print(f"Equity IRR = " , result["equity_irr"])
print(f"Quantity of lockups = ", result["lockup"].sum())
print(f"Quantity of defaults = ", result["default"].sum())

print("CASE WITH TARIFF = 100.77 AND L=8:")
sims = simulate(table, c, 1000, 8, np.random.default_rng(42))
sims.mean()

Table extracted per year: (30, 2)
Length of the final energy vector: 60

First 6 values of the vector (S1 year  1, S2 year 1, S1 year 2...):
[34.72624783 14.31234707 33.77413039 18.37214137 29.34512023 14.35140804]
Minimum DSCR (12m) =  0.9097530365619614
Equity IRR =  0.0777673413184432
Quantity of lockups =  3.0
Quantity of defaults =  2.0
CASE WITH TARIFF = 100.77 AND L=8:


min_dscr_12m       1.1409
any_lockup         0.6260
lockup_semesters   1.3870
any_default        0.3720
dsra_depleted      0.0210
sponsor_support    0.0000
equity_irr         0.0832
dtype: float64

**Interpretation**:
- In a "wet" year (a year with high precipitation/inflow), tax losses (carryforwards) are utilized more rapidly. The issue lies in the timing of tax payments: due to the FIFO (first-in, first-out) methodology, these tax losses are applied to offset taxes well before year 11, having been consumed earlier during the "wet" year. Consequently, from year 11 onwards, the project no longer has any tax credits (losses) available to offset taxes, forcing it to make tax payments. This reduces the CFADS (Cash Flow Available for Debt Service) and lowers the DSCR (Debt Service Coverage Ratio). Thus, while the project may generate significantly more total profit over its entire 30 year lifespan, a period-by-period analysis reveals these timing-related challenges.

**To measure the weight of the effect**:
- Another dictionary, 'c_notax', will be created in which the tax rate constant is disabled, changing it from 'tax_rate' = 0.35 to 'tax_rate' = 0.0.

In [7]:
c_100 = load_constants("../tests/fixtures/tariff100_p90.xlsx")

c_notax = dict(c_100)          # copy of the dictionary
c_notax["tax_rate"] = 0.0      # only this key changes; c_100 stays as it was

sims_notax = simulate(table, c_notax, 1000, 8, np.random.default_rng(42))
print("CASE WITH TARIFF = 100.77 AND NO TAX L=8:")
sims_notax.mean()

CASE WITH TARIFF = 100.77 AND NO TAX L=8:


min_dscr_12m         1.2410
any_lockup           0.4010
lockup_semesters     0.7330
any_default          0.2130
dsra_depleted        0.0210
sponsor_support    949.2432
equity_irr           0.1080
dtype: float64

**Interpretation**:
- It can be observed that the `any_default` rate is 21.3%, as detailed below. This confirms the effect of the tax-payment timing phenomenon we noted: with a 35% income tax, the probability of default at 100.77 USD/MWh is higher than in the base case (37.2% vs. 26.0%), whereas with taxes disabled, the probability of default is lower: 21.3% vs. 26.0%.
- It is important to note here that eliminating the taxes not only resolves the issue regarding the timing of tax payments, but the project also avoids paying the small amount of tax that the "sculpting" process had already accounted for; consequently, the project ends up with more cash on hand than the bank had projected.


Drought-related impact alone (excluding tax):    21.3%
Additional impact from tax timing:              + 15.9 percentage points
Total with 35% tax:                             37.2%


As a result, this gives us a double-edged interpretation:
- At a viable price point, the power plant withstands drought better (21% versus 26%), because fixed O&M costs have less of an impact.
- The repayment schedule structured for the P90 depends on when losses are incurred, and a "good" year can actually trigger a covenant breach years later. In practice, this is addressed through discussions with the bank (e.g., obtaining a waiver for the breach or restructuring the repayment schedule to provide more headroom).

**Hydrological scenario: recent period (2010–2025)**:
- The analysis of this case addresses the following question: if the climate conditions of the last 16 years were to persist, what level of risk would the loan (which was sized based on a 69-year period) carry?

- Next, it must be determined whether or not to change the value of L. Two perspectives can be considered:
    - Reducing the value of L, given that L=8 is quite large for a 16-year dataset (a criterion based on variety).
    - Maintaining the value of L, on the grounds that L represents the "memory" of the river—a characteristic that does not change simply because the time series does.
Neither alternative is superior to the other. In conclusion, a 16-year period is insufficient for a reliable bootstrap; therefore, this analysis of the recent period will be presented as a sensitivity test.

In [8]:
semi_recent = semi[semi["hydro_year"] >= 2010]
table_recent = semi_recent.pivot(index="hydro_year", columns="half", values="energy_base_gwh").to_numpy()
table_recent.shape      # should be (16, 2)

c_recent = load_constants("../model/hydro_pf_model_v1.xlsx")

# Simulate function with L=8
sims_recent_8 = simulate(table_recent, c_recent, 1000, 8, np.random.default_rng(42))
print("RECENT CASE WITH L=8:")
print(sims_recent_8.mean())
print()
# Simulate function with L=4
sims_recent_4 = simulate(table_recent, c_recent, 1000, 4, np.random.default_rng(42))
print("RECENT CASE WITH L=4: ")
print(sims_recent_4.mean())


RECENT CASE WITH L=8:
min_dscr_12m            1.1586
any_lockup              0.8040
lockup_semesters        1.6340
any_default             0.0000
dsra_depleted           0.4890
sponsor_support    21,683.7599
equity_irr              0.0192
dtype: float64

RECENT CASE WITH L=4: 
min_dscr_12m            1.1738
any_lockup              0.7750
lockup_semesters        1.6700
any_default             0.0000
dsra_depleted           0.4920
sponsor_support    21,720.9328
equity_irr              0.0192
dtype: float64


**Interpretation**:
- The recent period from 2010 to 2025 is mediocre, characterized by very low water flow, yet stable nonetheless. It is a period of "megadrought" where, despite the low flow rates, enough energy is still generated to avoid triggering a default.
- At 65 USD/MWh, the lock-up threshold occurs below approximately 42.8 GWh (95% of the P90 value), and the default threshold below approximately 40.5 GWh (90% of the P90 value). Three recent years fall within the "gray zone" (2011 at 42.5, 2020 at 42.0, and 2022 at 41.0), while none fall below 40.5. The two years from the full dataset that would have triggered a default—1969 (34.7) and 1971 (38.4), are not included in this sample.
- Warning: the fact that there hasn’t been an extreme year in 16 years does not prove that the recent climate is free of them.

- Final conclusion: a prolonged but moderate drought hinders dividend payouts (due to an 80% lock-up provision), while isolated extreme years lead to a default. This is a more nuanced version of H1.

**Hydrological scenario: E-FLOW 20%**:
- Now we analyze 20% e-flow scenario (as opposed to the base case, where e-flow = 10%) using its corresponding P90 debt figure (5.22 M). This approach assumes that the power plant is financed with the prior knowledge that the ecological flow will be 20% (rather than it changing at some point later). 

In [9]:

semi_20 = pd.read_csv("../data/processed/semiannual_energy.csv")
table_20 = semi_20.pivot(index="hydro_year", columns="half", values="energy_eco20_gwh").to_numpy()

c_20 = load_constants("../tests/fixtures/eco20_sized.xlsx")

# Simulate function with L=8
sims_20 = simulate(table_20, c_20, 1000, 8, np.random.default_rng(42))
print("E-FLOW 20% CASE WITH L=8:")
print(sims_20.mean())
print()



E-FLOW 20% CASE WITH L=8:
min_dscr_12m               1.0108
any_lockup                 0.7040
lockup_semesters           1.6320
any_default                0.6620
dsra_depleted              0.9900
sponsor_support    1,328,544.8128
equity_irr                 0.0183
dtype: float64



**Interpretation**:
- To exemplify the mechanism:

||Normal year |  Dry year  | Dry / Normal|
|:---------| :-------- | :-------- |  :-------- |
|E-flow 10% (−4,4) |   25,6    |     15,6    |     61%|
|E-flow 20% (−8,8)  |   21,2    |     11,2    |     53%|

- First effect: it is a matter of proportion: with a 20% e-flow, a dry year ends up being proportionally further from the norm. Sculpting compensates for the level (by providing less water), but not for the dispersion.
- Second effect: In the Excel file, the semi-annual P90 for all cases is distributed according to the seasonal profile of the base case (65.7% from October to March). However, the actual 20% e-flow series shows 71% occurring from October to March, based on that same proportional mechanism. Thus, the debt repayment schedule anticipates a higher energy output during dry semesters than the river actually provides, and the DSRA is drawn upon during nearly every dry semester.

|E-flow 20% |Real serie| With the excel's profile |
|:---------| :-------- | :-------- |
|P(default) |   66.2%    |     66.1%   | 
|P(depleted DSRA) |  99.0%    |     78.9%    |
|Sponsor support |  1.328.545    |     180.417    |

- P(default) remains unchanged, because the covenant considers a 12-month period, and within a 12-month window, the distribution between semesters effectively cancels out.
- DSRA and sponsor support does change: this highlights the limitations of stage 2.